In [2]:
import pandas as pd
import dash
from dash import dcc, html, Input, Output
import plotly.express as px

df = pd.read_csv("bbc_amharic_complete_20260624_024105.csv")

app = dash.Dash(__name__)

app.layout = html.Div([

    html.H1("BBC Amharic News Dashboard"),

    dcc.Dropdown(
        id="category-filter",
        options=[
            {"label": i, "value": i}
            for i in sorted(df["category_display"].dropna().unique())
        ],
        placeholder="Select Category"
    ),

    dcc.Dropdown(
        id="title-filter",
        options=[
            {"label": i[:80], "value": i}
            for i in df["title"].dropna().unique()
        ],
        placeholder="Select Title"
    ),

    dcc.Graph(id="news-chart")
])


@app.callback(
    Output("news-chart", "figure"),
    Input("category-filter", "value"),
    Input("title-filter", "value")
)
def update_chart(category, title):

    dff = df.copy()

    if category:
        dff = dff[dff["category_display"] == category]

    if title:
        dff = dff[dff["title"] == title]

    chart_data = (
        dff.groupby("category_display")
        .size()
        .reset_index(name="Articles")
    )

    fig = px.bar(
        chart_data,
        x="category_display",
        y="Articles",
        title="Number of Articles by Category"
    )

    return fig


if __name__ == "__main__":
    app.run(debug=True)

In [4]:
import pandas as pd
import dash
from dash import dcc, html, Input, Output
import plotly.express as px

df = pd.read_csv("bbc_amharic_complete_20260624_024105.csv")

app = dash.Dash(__name__)

app.layout = html.Div([

    html.H1("BBC Amharic News Dashboard"),

    dcc.Dropdown(
        id="category-filter",
        options=[
            {"label": c, "value": c}
            for c in sorted(df["category_display"].dropna().unique())
        ],
        placeholder="Select Category"
    ),

    dcc.Dropdown(
        id="title-filter",
        placeholder="Select Article"
    ),

    dcc.Graph(id="news-chart"),

    html.Hr(),

    html.H3("Article Details"),

    html.Div(id="article-content")
])


@app.callback(
    Output("title-filter", "options"),
    Input("category-filter", "value")
)
def update_titles(category):

    dff = df.copy()

    if category:
        dff = dff[dff["category_display"] == category]

    return [
        {"label": t[:100], "value": t}
        for t in dff["title"].dropna().unique()
    ]


@app.callback(
    Output("news-chart", "figure"),
    Input("category-filter", "value")
)
def update_chart(category):

    dff = df.copy()

    if category:
        dff = dff[dff["category_display"] == category]

    chart_data = (
        dff.groupby("category_display")
        .size()
        .reset_index(name="Articles")
    )

    fig = px.bar(
        chart_data,
        x="category_display",
        y="Articles",
        title="Articles by Category"
    )

    return fig


@app.callback(
    Output("article-content", "children"),
    Input("title-filter", "value")
)
def show_article(title):

    if not title:
        return "Select an article to view its content."

    row = df[df["title"] == title].iloc[0]

    return html.Div([
        html.H4(row["title"]),
        html.P(f"Date: {row['date']}"),
        html.H5("Summary"),
        html.P(row["summary"]),
        html.H5("Full Content"),
        html.P(row["full_content"]),
        html.Br(),
        html.A("Read Original Article", href=row["link"], target="_blank")
    ])


if __name__ == "__main__":
    app.run(debug=True)